## Visualização dos Dados

In [ ]:
# Imports
import mne
import numpy as np
import pandas as pd
from pathlib import Path
import sys

# Add the Inner Speech Dataset repository to the system path
sys.path.append(str(Path.cwd() / "Python_Processing"))

# Import custom functions from the Inner Speech Dataset repository
from lib.data_extractions import (
    extract_data_from_subject,
    load_events
)

from lib.data_processing import (
    select_time_window,
    transform_for_classificator,
)

from lib.download_helper import (
    _clone_inner_speech,
    get_derivatives,
)

from lib.utils import (
    sub_name, 
    unify_names,
)

# List with every subject (S1-S10)
N_S = list(range(1, 11))

# sub-09 ses-01 and sub-10 ses-03 are truncated in the OpenNeuro datasets
EXCLUDE = {9, 10}
SUBJECTS = [i for i in N_S if i not in EXCLUDE]
CHANNELS = {''}

data_dir = Path.cwd().parent
root_dir = data_dir / "ds003626"

In [ ]:
import matplotlib.pyplot as plt

CLASSES = {0: "up", 1: "down", 2: "right", 3: "left"}

times = np.load("data/S1_times.npy")
ch_names = np.load("data/S1_channels.npy")
X_cls = np.load("data/X_cls.npy", mmap_mode="r")

# x = np.asarray(X_cls[0, :50])
# print("std:", x.std(), "| max abs:", np.abs(x).max())

subj = np.load("data/subj.npy")

print(times[0], times[-1], len(times)) # check time range

def plot_channels(classes=(0, 1, 2, 3), subject=None, tmin=None, tmax=None, ncols=8, save_as=None):
    keep = np.ones(len(times), dtype=bool)

    if tmin is not None: keep &= times >= tmin
    if tmax is not None: keep &= times <= tmax
    
    t = times[keep]
    n_ch = X_cls.shape[2]
    nrows = int(np.ceil(n_ch / ncols))
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, 2.0 * nrows), sharex=True, sharey=True)

    for c in classes:
        trials = X_cls[c] if subject is None else X_cls[c][subj[c] == subject]
        mean = trials[:, :, keep].mean(axis=0) * 1e6 # convert volts to microvolts
        
        for ch, ax in enumerate(axes.flat[:n_ch]):
            ax.plot(t, mean[ch], lw=0.3, label=CLASSES[c])

    for ch, ax in enumerate(axes.flat[:n_ch]):
        ax.set_title(str(ch_names[ch]), fontsize=7, pad=2)
        ax.tick_params(labelsize=6)
        ax.axvline(0, color="black", lw=0.4, alpha=0.4)
    
    for ax in axes.flat[n_ch:]:
        ax.axis("off")

    # axes.flat[0].legend(fontsize=6, loc="upper right")
    sbj = "all subjects" if subject is None else f"subject {subject}"
    fig.legend(
        labels=[CLASSES[c] for c in classes],
        loc="upper center",
        bbox_to_anchor=(0.5, 1.0),
        ncol=len(classes),
        fontsize=10,
        frameon=False
    )
    fig.suptitle(f"Mean EEG per channel ({sbj}), µV vs. time (s)", y=1.002)
    fig.tight_layout()

    if save_as:
        fig.savefig(save_as, dpi=600, bbox_inches="tight")
    
    plt.show()

In [ ]:
for i in SUBJECTS:
    plot_channels(subject=i, save_as=f"figures/s{i}")

In [ ]:
# Debugging
X_cls = np.load("data/X_cls.npy")
subj = np.load("data/subj.npy")
ses = np.load("data/ses.npy")

print(X_cls.shape, subj.shape, ses.shape) # expected (4, 439, 128, 1153) (4, 439) (4, 439)
print(np.unique(subj[0], return_counts=True)) # expected 50 60 45 60 60 54 60 50 for subjects 1, 2, ..., 8
print(np.unique(ses, return_counts=True)) # sessions 1, 2, 3 only
print((subj == subj[0]).all()) # same subject order in every class